## Cell 1 — Install Ultralytics + SAHI

In [ ]:

# !pip install -q ultralytics sahi


## Cell 2 — Paths (reuse the same YOLO dataset you already built)

In [2]:

import os

BASE_DIR = r"C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone"
YOLO_DATASET_DIR = os.path.join(BASE_DIR, "yolo_full_dataset")
data_yaml_path = os.path.join(YOLO_DATASET_DIR, "data.yaml")

print("Using existing dataset config:", data_yaml_path)
print("Exists?", os.path.exists(data_yaml_path))


Using existing dataset config: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\data.yaml
Exists? True


## Cell 3 — Create a P2-enabled YOLOv11 model config\n\nStandard YOLOv11 detects at P3/P4/P5 (strides 8/16/32). Adding a P2 head (stride 4) gives a higher-resolution detection layer, which is proven to boost small-object recall (this is exactly the 'P2 layer' trick the DFFormer paper's own Table VI compares against, giving DEIM +2.3 mAP over baseline).

In [3]:

import ultralytics, os

# Locate Ultralytics' default yolo11 config to use as a base
ultra_path = os.path.dirname(ultralytics.__file__)
base_cfg_path = os.path.join(ultra_path, "cfg", "models", "11", "yolo11.yaml")

with open(base_cfg_path, "r") as f:
    print(f.read())


# Ultralytics ðŸš€ AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 80 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

# YOLO11n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 

## Cell 4 — Write a custom yolo11-p2.yaml with an added P2 detection head

In [4]:

p2_yaml_content = """
# YOLOv11 with added P2 (stride 4) detection head for small-object detection
nc: 1  # number of classes (UAV)
scales:
  s: [0.50, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]        # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]       # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]       # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]       # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]      # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]         # 9
  - [-1, 2, C2PSA, [1024]]           # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]      # 13 (P4)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]      # 16 (P3)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, False]]      # 19 (P2, NEW - stride 4, small object head)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 16], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]      # 22 (P3)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]      # 25 (P4)

  - [-1, 1, Conv, [1024, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]      # 28 (P5)

  - [[19, 22, 25, 28], 1, Detect, [nc]]  # Detect(P2, P3, P4, P5)
"""

p2_yaml_path = os.path.join(YOLO_DATASET_DIR, "yolo11-p2.yaml")
with open(p2_yaml_path, "w") as f:
    f.write(p2_yaml_content)

print("Saved custom P2 model config to:", p2_yaml_path)


Saved custom P2 model config to: C:\Users\HP\OneDrive\Documents\Desktop\vlm\data\detection_drone\yolo_full_dataset\yolo11-p2.yaml


## Cell 5 — Train YOLOv11 with the P2 head

In [9]:

from ultralytics import YOLO

model = YOLO(p2_yaml_path)  # build from custom architecture (random init)
# Load pretrained yolo11s weights into matching layers for transfer learning
model = model.load("yolo11s.pt")

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=960,
    batch=4,
    device=0,
    patience=10,
    project=os.path.join(BASE_DIR, "yolo_runs"),
    name="uav_detect_p2",
    exist_ok=True,
    seed=42,
)


WARNING no model scale passed. Assuming scale='s'.
Transferred 297/593 items from pretrained weights
Ultralytics 8.4.92  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Cell 6 — Evaluate P2 model (compare directly against your existing yolo11s run)

In [ ]:

best_weights_p2 = os.path.join(BASE_DIR, "yolo_runs", "uav_detect_p2", "weights", "best.pt")
print("Using weights:", best_weights_p2)

trained_model_p2 = YOLO(best_weights_p2)
metrics_p2 = trained_model_p2.val(data=data_yaml_path, imgsz=960, device=0)

print("\n===== YOLOv11 + P2 Head Evaluation Summary =====")
print(f"Precision:   {metrics_p2.box.mp:.4f}")
print(f"Recall:      {metrics_p2.box.mr:.4f}")
print(f"mAP50:       {metrics_p2.box.map50:.4f}")
print(f"mAP50-95:    {metrics_p2.box.map:.4f}")


## Cell 7 — SAHI tiled inference setup\n\nSAHI slices each image into overlapping tiles, runs detection on each tile, then merges results back to full-image coordinates. This directly compensates for small objects becoming even smaller after internal model resizing -- a well-established boost for tiny-object detection, no retraining needed.

In [ ]:

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",  # SAHI's ultralytics wrapper also covers yolo11 checkpoints
    model_path=best_weights_p2,
    confidence_threshold=0.25,
    device="cuda:0",
)

print("SAHI model loaded from:", best_weights_p2)


## Cell 8 — Run SAHI sliced prediction on a few val images and visualize

In [ ]:

import matplotlib.pyplot as plt

YOLO_IMAGES_VAL = os.path.join(YOLO_DATASET_DIR, "images", "val")
val_images = sorted(os.listdir(YOLO_IMAGES_VAL))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, fname in enumerate(val_images):
    img_path = os.path.join(YOLO_IMAGES_VAL, fname)

    result = get_sliced_prediction(
        img_path,
        sahi_model,
        slice_height=480,
        slice_width=480,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
    )

    result.export_visuals(export_dir=os.path.join(BASE_DIR, "sahi_preview"), file_name=f"pred_{i}")
    preview_path = os.path.join(BASE_DIR, "sahi_preview", f"pred_{i}.png")

    img = plt.imread(preview_path)
    axes[i].imshow(img)
    axes[i].set_title(fname)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## Cell 9 — Save comparison summary (yolo11s baseline vs P2 head vs P2+SAHI)

In [ ]:

from datetime import datetime

report_path = os.path.join(BASE_DIR, "p2_sahi_comparison_summary.txt")

with open(report_path, "w") as f:
    f.write("YOLOv11 Small-Object Detection Improvements - Comparison Report\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n\n")
    f.write("===== YOLOv11s + P2 Head (trained, no tiling) =====\n")
    f.write(f"Precision:   {metrics_p2.box.mp:.4f}\n")
    f.write(f"Recall:      {metrics_p2.box.mr:.4f}\n")
    f.write(f"mAP50:       {metrics_p2.box.map50:.4f}\n")
    f.write(f"mAP50-95:    {metrics_p2.box.map:.4f}\n\n")
    f.write("NOTE: Compare these numbers against your earlier plain yolo11s run\n")
    f.write("(without P2) to quantify the improvement from adding the high-\n")
    f.write("resolution detection head. SAHI tiled inference (Cell 7-8) further\n")
    f.write("improves recall on small/tiny UAV instances at inference time,\n")
    f.write("independent of the training process -- run a full SAHI evaluation\n")
    f.write("pass (not just visualization) if you want a quantified mAP for it too.\n")

print("Summary saved to:", report_path)
